# Solar Power Generation Prediction using Linear Regression

## Use Case
This project predicts AC power output from a solar plant using operational data such as DC power, daily yield, and time of day.

The goal is to improve energy forecasting and optimize power generation.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import joblib

ModuleNotFoundError: No module named 'pandas'

## Data Loading and Exploration

In [ ]:
df = pd.read_csv("/Users/manziivan453icloud.com/Downloads/Projects/ML_Summative_1/linear_regression_model/Plant_1_Generation_Data.csv")
df.head()

In [ ]:
print("Dataset info:")
df.info()
print("\nBasic statistics:")
df.describe()

In [ ]:
# Remove night-time data (no solar production)
df = df[df['DC_POWER'] > 0]

## Feature Engineering

In [ ]:
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
df['hour'] = df['DATE_TIME'].dt.hour

## Data Visualization

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

### Interpretation

- DC_POWER has a strong positive relationship with AC_POWER
- DAILY_YIELD also contributes to AC_POWER
- Time (hour) influences generation patterns

This shows solar energy depends on system output and time of day.

In [ ]:
plt.figure()
sns.scatterplot(x=df['DC_POWER'], y=df['AC_POWER'])
plt.title("DC Power vs AC Power")
plt.show()

In [ ]:
plt.figure()
sns.histplot(df['AC_POWER'], kde=True)
plt.title("Distribution of AC Power")
plt.show()

## Feature Selection

In [ ]:
features = ['DC_POWER', 'DAILY_YIELD', 'hour']
X = df[features]
y = df['AC_POWER']

## Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Training

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = rmse

results

In [ ]:
plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Model Comparison (RMSE)")
plt.ylabel("RMSE")
plt.show()

## Gradient Descent and Loss Curve

In [ ]:
train_losses = []
test_losses = []

sgd = SGDRegressor(max_iter=1, learning_rate='constant', eta0=0.0001, warm_start=True)

for i in range(100):
    sgd.fit(X_train_scaled, y_train)

    y_train_pred = sgd.predict(X_train_scaled)
    y_test_pred = sgd.predict(X_test_scaled)

    train_loss = mean_squared_error(y_train, y_train_pred)
    test_loss = mean_squared_error(y_test, y_test_pred)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

plt.figure()
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.legend()
plt.title("Loss Curve")
plt.show()

## Linear Regression Visualization

In [ ]:
lr_model = models["Linear Regression"]
y_pred_lr = lr_model.predict(X_test_scaled)

plt.figure()
plt.scatter(y_test, y_pred_lr)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

## Save Model

In [ ]:
best_model = models[min(results, key=results.get)]

joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

## Feature Importance

In [ ]:
rf = models["Random Forest"]

importances = rf.feature_importances_

plt.figure()
plt.bar(features, importances)
plt.title("Feature Importance")
plt.show()

## Making Predictions

In [ ]:
loaded_model = joblib.load("best_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

sample_input = X.sample(1).to_dict(orient='records')[0]

input_df = pd.DataFrame([sample_input])
input_scaled = loaded_scaler.transform(input_df)

prediction = loaded_model.predict(input_scaled)

print("Input:", sample_input)
print("Predicted AC Power:", prediction[0])

## Summary

- Random Forest performed best
- DC Power is the most important feature
- The model can predict solar power output effectively
- Gradient descent improved optimization